[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/072.%20The%20Capacitated%20VRP%20%28CVRP%29/P72-Tier-5_executed.ipynb)

---


# 72. The Capacitated Vehicle Routing Problem (CVRP)

## Tier 5: Digital-Twin Concepts and Real-Time Synchronization

### Purpose and scope

This notebook illustrates how a digital-twin architecture can organize sensor data, predictive analytics, and what-if scenario tests around a vehicle-routing operation. It uses simulated data and simplified models to make the information flow inspectable. It does **not** formulate or solve a capacitated vehicle routing problem: routes are supplied as input, vehicle capacities are not enforced, and the scenario calculations are illustrative rather than operational forecasts.

A digital twin is a software representation of a physical system that is updated using observations from that system. Here, sensors, data streams, predictive-model records, and scenario tests are represented as Python objects. The notebook simulates their interactions; it does not connect to physical devices or external data services.

### Inputs and outputs

- **Inputs:** the configured sensor network, simulated sensor measurements, four example routes, and illustrative scenario parameters.
- **Outputs:** sensor and synchronization summaries, demand and fuel estimates, route-ordering demonstrations, scenario metrics, a text report, and a visualization.
- **Interpretation:** reported figures describe this simulation and its configured assumptions. They are not measured service levels, validated model accuracy, or evidence of real-world improvement.

### Modeling assumptions

- Sensor measurements are generated synthetically. Random draws use the notebook's fixed seed; the demand forecast uses a fixed simulation hour so that its time pattern is reproducible.
- Sensor locations and route distances are not derived from a road network. For the route-ordering demonstration only, customer identifiers are treated as positions on a one-dimensional line, with the depot at position 0.
- The route-ordering demonstration applies the existing 2-opt local-search procedure to that one-dimensional proxy. It does not impose CVRP capacity constraints or reassign customers between vehicles.
- Scenario results use simple arithmetic rules and illustrative baseline metrics. They do not simulate vehicle movement, solve a routing model, or establish causal effects.
- Values stored as model accuracy or system uptime are configured example values unless explicitly calculated from simulated observations. They are not independently validated performance measurements.

### Learning sequence

The notebook first defines the twin's data structures, then generates and integrates sensor data, records predictive estimates, tests scenarios, and assembles a report and plots. Each stage contributes a different kind of information; none by itself constitutes a complete CVRP optimization model.

In [ ]:
from dataclasses import dataclass
from enum import Enum
import random
import time
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np

# A fixed seed makes synthetic random measurements repeatable across runs.
np.random.seed(42)
random.seed(42)

print('Libraries imported for the digital-twin simulation.')

In [ ]:
class SensorType(Enum):
    GPS = 'gps'
    TEMPERATURE = 'temperature'
    HUMIDITY = 'humidity'
    VIBRATION = 'vibration'
    FUEL_LEVEL = 'fuel_level'
    LOAD_WEIGHT = 'load_weight'
    TRAFFIC = 'traffic'
    WEATHER = 'weather'


class DataStreamType(Enum):
    REAL_TIME = 'real_time'
    BATCH = 'batch'
    STREAMING = 'streaming'
    PREDICTIVE = 'predictive'


@dataclass
class IoTSensor:
    """A sensor record and the measurements generated for it."""

    sensor_id: str
    sensor_type: SensorType
    location: str
    sampling_rate: float  # Samples per second (Hz).
    accuracy: float  # Configured value from 0.0 to 1.0; not an empirical test.
    battery_life: float  # Hours.
    data_points: List[Dict[str, Any]]

    def __post_init__(self) -> None:
        self.last_update = time.time()
        self.is_active = True
        self.maintenance_schedule = 30 * 24  # Hours in a 30-day interval.


@dataclass
class DataStream:
    """A configured channel that groups measurements from sensors."""

    stream_id: str
    stream_type: DataStreamType
    source_sensors: List[str]
    data_rate: float  # Configured points per second.
    latency: float  # Configured milliseconds.
    reliability: float  # Configured value from 0.0 to 1.0.

    def __post_init__(self) -> None:
        self.buffer_size = 1000
        self.data_buffer: List[Dict[str, Any]] = []
        self.last_processed = time.time()


@dataclass
class PredictiveModel:
    """Metadata and illustrative performance records for a model."""

    model_id: str
    model_type: str
    accuracy: float
    update_frequency: float  # Hours.
    training_data_size: int

    def __post_init__(self) -> None:
        self.last_training = time.time()
        self.prediction_history: List[Any] = []
        self.performance_metrics = {'mae': 0.0, 'rmse': 0.0, 'r2_score': 0.0}


@dataclass
class ScenarioTest:
    """Parameters, baseline, and results for one illustrative scenario."""

    test_id: str
    scenario_name: str
    parameters: Dict[str, Any]
    baseline_metrics: Dict[str, float]
    test_results: Dict[str, float]
    improvement_percentage: float

    def __post_init__(self) -> None:
        self.test_duration = 0.0
        self.confidence_level = 0.95


@dataclass
class DigitalTwin:
    """A container linking the virtual components of the demonstration."""

    twin_id: str
    physical_system_id: str
    synchronization_rate: float  # Hz.
    fidelity_level: float  # Configured value from 0.0 to 1.0.
    uptime: float  # Configured percentage, not measured uptime.

    def __post_init__(self) -> None:
        self.sensors: List[IoTSensor] = []
        self.data_streams: List[DataStream] = []
        self.predictive_models: List[PredictiveModel] = []
        self.scenario_tests: List[ScenarioTest] = []
        self.performance_metrics = {
            'sync_accuracy': 0.0,
            'data_throughput': 0.0,
            'prediction_accuracy': 0.0,
            'scenario_test_speed': 0.0,
        }
        self.system_health = {
            'sensor_status': 'healthy',
            'data_quality': 'simulated',
            'model_performance': 'configured',
            'overall_status': 'simulation',
        }

print('Digital-twin data structures defined.')

In [ ]:
class IoTSensorManager:
    """Creates a sensor network and generates synthetic measurements."""

    def __init__(self) -> None:
        self.sensors: List[IoTSensor] = []
        self.sensor_network: Dict[str, IoTSensor] = {}
        self.generated_duration_minutes = 0
        self._initialize_sensors()
        self.sensor_network = {sensor.sensor_id: sensor for sensor in self.sensors}

    def _initialize_sensors(self) -> None:
        # Vehicle telemetry includes position, fuel, and load measurements.
        for vehicle_id in range(1, 5):
            self.sensors.extend([
                IoTSensor(
                    sensor_id=f'gps_vehicle_{vehicle_id}',
                    sensor_type=SensorType.GPS,
                    location=f'Vehicle_{vehicle_id}',
                    sampling_rate=1.0,
                    accuracy=0.98,
                    battery_life=720.0,
                    data_points=[],
                ),
                IoTSensor(
                    sensor_id=f'fuel_vehicle_{vehicle_id}',
                    sensor_type=SensorType.FUEL_LEVEL,
                    location=f'Vehicle_{vehicle_id}',
                    sampling_rate=0.1,
                    accuracy=0.95,
                    battery_life=2160.0,
                    data_points=[],
                ),
                IoTSensor(
                    sensor_id=f'load_vehicle_{vehicle_id}',
                    sensor_type=SensorType.LOAD_WEIGHT,
                    location=f'Vehicle_{vehicle_id}',
                    sampling_rate=0.5,
                    accuracy=0.97,
                    battery_life=1080.0,
                    data_points=[],
                ),
            ])

        # Weather and traffic are represented at three illustrative locations.
        for location_id in range(1, 4):
            self.sensors.extend([
                IoTSensor(
                    sensor_id=f'weather_loc_{location_id}',
                    sensor_type=SensorType.WEATHER,
                    location=f'Location_{location_id}',
                    sampling_rate=0.01,
                    accuracy=0.92,
                    battery_life=4320.0,
                    data_points=[],
                ),
                IoTSensor(
                    sensor_id=f'traffic_loc_{location_id}',
                    sensor_type=SensorType.TRAFFIC,
                    location=f'Location_{location_id}',
                    sampling_rate=0.2,
                    accuracy=0.89,
                    battery_life=2880.0,
                    data_points=[],
                ),
            ])

        # Customer GPS sensors provide synthetic customer-location observations.
        for customer_id in range(1, 11):
            self.sensors.append(
                IoTSensor(
                    sensor_id=f'customer_{customer_id}',
                    sensor_type=SensorType.GPS,
                    location=f'Customer_{customer_id}',
                    sampling_rate=0.05,
                    accuracy=0.96,
                    battery_life=1440.0,
                    data_points=[],
                )
            )

    def _generate_sensor_value(self, sensor: IoTSensor, timestamp: float) -> Dict[str, Any]:
        """Draw one synthetic observation with fields appropriate to its sensor."""
        if sensor.sensor_type == SensorType.GPS:
            return {
                'timestamp': timestamp,
                'latitude': float(np.random.uniform(40.7, 40.8)),
                'longitude': float(np.random.uniform(-74.0, -73.9)),
                'accuracy': sensor.accuracy * float(np.random.uniform(0.95, 1.0)),
            }
        if sensor.sensor_type == SensorType.FUEL_LEVEL:
            return {
                'timestamp': timestamp,
                'fuel_percentage': float(np.clip(np.random.normal(50, 20), 0, 100)),
                'consumption_rate': float(np.random.uniform(0.1, 0.5)),
            }
        if sensor.sensor_type == SensorType.LOAD_WEIGHT:
            return {
                'timestamp': timestamp,
                'current_load': float(np.random.uniform(0, 1000)),
                'capacity_utilization': float(np.random.uniform(0, 1)),
            }
        if sensor.sensor_type == SensorType.WEATHER:
            return {
                'timestamp': timestamp,
                'temperature': float(np.random.uniform(-10, 35)),
                'humidity': float(np.random.uniform(20, 90)),
                'wind_speed': float(np.random.uniform(0, 20)),
            }
        if sensor.sensor_type == SensorType.TRAFFIC:
            return {
                'timestamp': timestamp,
                'traffic_level': str(np.random.choice(['light', 'moderate', 'heavy'])),
                'congestion_index': float(np.random.uniform(0, 1)),
            }
        return {'timestamp': timestamp, 'value': float(np.random.uniform(0, 100))}

    def generate_sensor_data(self, duration_minutes: int = 10) -> int:
        """Generate measurements at each sensor's configured sampling rate."""
        if duration_minutes <= 0:
            raise ValueError('duration_minutes must be positive.')

        start_time = time.time()
        duration_seconds = duration_minutes * 60
        generated_count = 0
        self.generated_duration_minutes = duration_minutes

        for sensor in self.sensors:
            # Evenly spaced timestamps preserve the configured average sampling rate,
            # including rates that produce fewer than one sample in a single minute.
            number_of_points = int(sensor.sampling_rate * duration_seconds)
            for point_index in range(number_of_points):
                timestamp = start_time + point_index / sensor.sampling_rate
                sensor.data_points.append(self._generate_sensor_value(sensor, timestamp))
                generated_count += 1
            if sensor.data_points:
                sensor.last_update = sensor.data_points[-1]['timestamp']

        return generated_count

    def get_sensor_statistics(self) -> Dict[str, Any]:
        """Summarize sensor availability, configured accuracy, and generated volume."""
        total_sensors = len(self.sensors)
        active_sensors = sum(sensor.is_active for sensor in self.sensors)
        total_data_points = sum(len(sensor.data_points) for sensor in self.sensors)
        average_accuracy = float(np.mean([sensor.accuracy for sensor in self.sensors]))
        sensor_types: Dict[str, int] = {}
        for sensor in self.sensors:
            name = sensor.sensor_type.value
            sensor_types[name] = sensor_types.get(name, 0) + 1

        duration_seconds = self.generated_duration_minutes * 60
        points_per_minute = (
            total_data_points / self.generated_duration_minutes
            if self.generated_duration_minutes
            else 0.0
        )
        return {
            'total_sensors': total_sensors,
            'active_sensors': active_sensors,
            'total_data_points': total_data_points,
            'average_accuracy': average_accuracy,
            'sensor_types': sensor_types,
            'data_points_per_minute': points_per_minute,
            'generated_duration_seconds': duration_seconds,
        }

print('IoT sensor management system defined.')

In [ ]:
class RealTimeDataIntegration:
    """Groups the latest simulated sensor observations into a twin snapshot."""

    def __init__(self, sensor_manager: IoTSensorManager) -> None:
        self.sensor_manager = sensor_manager
        self.data_streams: List[DataStream] = []
        self.integration_buffer: List[Dict[str, Any]] = []
        self._initialize_data_streams()

    def _initialize_data_streams(self) -> None:
        sensors = self.sensor_manager.sensors
        vehicle_sensors = [s.sensor_id for s in sensors if 'vehicle' in s.sensor_id]
        environmental_sensors = [
            s.sensor_id for s in sensors
            if s.sensor_type in (SensorType.WEATHER, SensorType.TRAFFIC)
        ]
        customer_sensors = [s.sensor_id for s in sensors if 'customer' in s.sensor_id]

        self.data_streams = [
            DataStream('vehicle_telemetry', DataStreamType.REAL_TIME,
                       vehicle_sensors, 25.0, 50.0, 0.98),
            DataStream('environmental_data', DataStreamType.STREAMING,
                       environmental_sensors, 5.0, 100.0, 0.95),
            DataStream('customer_data', DataStreamType.BATCH,
                       customer_sensors, 2.0, 200.0, 0.96),
        ]

    def process_real_time_data(self) -> Dict[str, Any]:
        """Build a snapshot from each sensor's most recent available observation."""
        processed_data: Dict[str, Any] = {
            'timestamp': time.time(),
            'vehicle_positions': {},
            'fuel_levels': {},
            'load_weights': {},
            'environmental_conditions': {},
            'customer_locations': {},
            'data_quality_metrics': {},
        }

        for sensor in self.sensor_manager.sensors:
            if not sensor.data_points:
                continue
            latest = sensor.data_points[-1]
            if 'vehicle' in sensor.sensor_id:
                vehicle_id = sensor.sensor_id.split('_')[-1]
                if sensor.sensor_type == SensorType.GPS:
                    processed_data['vehicle_positions'][vehicle_id] = {
                        'lat': latest['latitude'], 'lon': latest['longitude'],
                        'accuracy': latest['accuracy'],
                    }
                elif sensor.sensor_type == SensorType.FUEL_LEVEL:
                    processed_data['fuel_levels'][vehicle_id] = {
                        'percentage': latest['fuel_percentage'],
                        'consumption_rate': latest['consumption_rate'],
                    }
                elif sensor.sensor_type == SensorType.LOAD_WEIGHT:
                    processed_data['load_weights'][vehicle_id] = {
                        'current_load': latest['current_load'],
                        'utilization': latest['capacity_utilization'],
                    }
            elif sensor.sensor_type in (SensorType.WEATHER, SensorType.TRAFFIC):
                location_id = sensor.sensor_id.split('_')[-1]
                conditions = processed_data['environmental_conditions'].setdefault(
                    location_id, {}
                )
                if sensor.sensor_type == SensorType.WEATHER:
                    conditions['weather'] = {
                        'temperature': latest['temperature'],
                        'humidity': latest['humidity'],
                        'wind_speed': latest['wind_speed'],
                    }
                else:
                    conditions['traffic'] = {
                        'level': latest['traffic_level'],
                        'congestion': latest['congestion_index'],
                    }
            elif 'customer' in sensor.sensor_id:
                customer_id = sensor.sensor_id.split('_')[-1]
                processed_data['customer_locations'][customer_id] = {
                    'lat': latest['latitude'],
                    'lon': latest['longitude'],
                    'accuracy': latest['accuracy'],
                }

        total_sensors = len(self.sensor_manager.sensors)
        active_sensors = sum(sensor.is_active for sensor in self.sensor_manager.sensors)
        sensors_with_data = sum(bool(sensor.data_points) for sensor in self.sensor_manager.sensors)
        processed_data['data_quality_metrics'] = {
            'sensor_availability': active_sensors / total_sensors if total_sensors else 0.0,
            'data_completeness': sensors_with_data / total_sensors if total_sensors else 0.0,
            'average_configured_accuracy': float(
                np.mean([sensor.accuracy for sensor in self.sensor_manager.sensors])
            ) if total_sensors else 0.0,
            'configured_mean_latency_ms': float(
                np.mean([stream.latency for stream in self.data_streams])
            ) if self.data_streams else 0.0,
        }
        self.integration_buffer.append(processed_data)
        return processed_data

    def calculate_synchronization_metrics(self) -> Dict[str, float]:
        """Report simulated volume and configured stream properties with units."""
        stats = self.sensor_manager.get_sensor_statistics()
        sensors = self.sensor_manager.sensors
        streams = self.data_streams
        return {
            'data_throughput': stats['data_points_per_minute'],
            'sync_accuracy': float(np.mean([s.accuracy for s in sensors])) if sensors else 0.0,
            'system_reliability': (
                sum(s.is_active for s in sensors) / len(sensors) if sensors else 0.0
            ),
            'avg_latency': float(np.mean([stream.latency for stream in streams])) if streams else 0.0,
            'uptime': 99.7,
        }

print('Real-time data integration system defined.')

In [ ]:
class PredictiveAnalyticsSystem:
    """Produces illustrative forecasts and a local route-ordering demonstration."""

    def __init__(self) -> None:
        self.models: List[PredictiveModel] = []
        self._initialize_models()

    def _initialize_models(self) -> None:
        self.models = [
            PredictiveModel('demand_forecast_v1', 'time_series_example', 0.94, 1.0, 10000),
            PredictiveModel('route_optimizer_v1', '2-opt_local_search', 0.87, 2.0, 50000),
            PredictiveModel('fuel_consumption_v1', 'regression_example', 0.91, 4.0, 25000),
            PredictiveModel('traffic_predictor_v1', 'neural_network_example', 0.83, 0.5, 15000),
        ]

    def predict_demand(
        self, historical_data: List[Dict[str, Any]], forecast_horizon: int = 24
    ) -> Dict[str, Any]:
        """Create a synthetic hourly demand pattern; no historical data is fitted."""
        if forecast_horizon <= 0:
            raise ValueError('forecast_horizon must be positive.')
        # A fixed simulation hour separates this repeatable example from wall-clock time.
        simulation_hour = 0.0
        base_demand = 50.0
        predictions = []
        for hour in range(forecast_horizon):
            hour_factor = 1.0 + 0.3 * np.sin(
                2 * np.pi * (simulation_hour + hour) / 24
            )
            noise = np.random.normal(0, 5)
            predictions.append(float(max(10, base_demand * hour_factor + noise)))
        intervals = [(prediction * 0.9, prediction * 1.1) for prediction in predictions]
        return {
            'predictions': predictions,
            'confidence_intervals': intervals,
            'model_accuracy': 0.94,
            'forecast_horizon': forecast_horizon,
        }

    def predict_optimal_routes(
        self, current_routes: List[List[int]], demand_forecast: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Apply 2-opt to each supplied route using a one-dimensional distance proxy."""
        optimized_routes: List[List[int]] = []
        improvements: List[float] = []
        for route in current_routes:
            optimized_route = self._two_opt_optimization(route)
            original_distance = self._calculate_route_distance(route)
            optimized_distance = self._calculate_route_distance(optimized_route)
            improvement = (
                (original_distance - optimized_distance) / original_distance
                if original_distance > 0 else 0.0
            )
            optimized_routes.append(optimized_route)
            improvements.append(float(improvement))
        return {
            'optimized_routes': optimized_routes,
            'improvements': improvements,
            'average_improvement': float(np.mean(improvements)) if improvements else 0.0,
            'model_accuracy': 0.87,
        }

    def predict_fuel_consumption(
        self, routes: List[List[int]], weather_conditions: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Estimate fuel with a simple per-customer and weather multiplier."""
        predictions = []
        for route in routes:
            base_consumption = len(route) * 2.5
            temperature = weather_conditions.get('temperature', 20)
            wind = weather_conditions.get('wind_speed', 5)
            weather_factor = (
                1.0 + 0.1 * (abs(temperature - 20) / 20) + 0.05 * (wind / 20)
            )
            congestion = weather_conditions.get('congestion', 0.5)
            traffic_factor = 1.0 + 0.2 * congestion
            predictions.append(base_consumption * weather_factor * traffic_factor)
        return {
            'fuel_predictions': predictions,
            'total_consumption': sum(predictions),
            'model_accuracy': 0.91,
        }

    def _two_opt_optimization(self, route: List[int]) -> List[int]:
        """Search route reversals while retaining the supplied customer set."""
        if len(route) <= 2:
            return route.copy()
        best_route = route.copy()
        best_distance = self._calculate_route_distance(best_route)
        improved = True
        iterations = 0
        max_iterations = min(50, len(route) * 5)
        while improved and iterations < max_iterations:
            improved = False
            iterations += 1
            for i in range(len(best_route) - 1):
                for j in range(i + 2, len(best_route)):
                    candidate = (
                        best_route[:i + 1]
                        + best_route[i + 1:j + 1][::-1]
                        + best_route[j + 1:]
                    )
                    candidate_distance = self._calculate_route_distance(candidate)
                    if candidate_distance < best_distance:
                        best_route = candidate
                        best_distance = candidate_distance
                        improved = True
        return best_route

    def _calculate_route_distance(self, route: List[int]) -> float:
        """Measure a route on a line, with customer ID as position and depot at 0."""
        if not route:
            return 0.0
        positions = [0, *route, 0]
        return float(sum(
            abs(positions[index + 1] - positions[index])
            for index in range(len(positions) - 1)
        ))

    def get_model_performance(self) -> Dict[str, Any]:
        """Return configured model records, not validation results."""
        return {
            model.model_id: {
                'model_type': model.model_type,
                'accuracy': model.accuracy,
                'update_frequency': model.update_frequency,
                'training_data_size': model.training_data_size,
                'performance_metrics': model.performance_metrics,
            }
            for model in self.models
        }

print('Predictive analytics system defined.')

In [ ]:
class ScenarioTestingSystem:
    """Applies transparent arithmetic rules to illustrative what-if cases."""

    def __init__(self) -> None:
        self.scenarios: List[ScenarioTest] = []
        self.test_results: List[Dict[str, Any]] = []
        self._initialize_scenarios()

    def _initialize_scenarios(self) -> None:
        baseline = {
            'total_distance': 100.0,
            'delivery_time': 120.0,
            'fuel_consumption': 50.0,
            'vehicle_utilization': 0.75,
        }
        self.scenarios = [
            ScenarioTest(
                'demand_surge_001', 'Demand Surge Scenario',
                {'demand_multiplier': 1.5, 'duration_hours': 4,
                 'affected_areas': ['downtown', 'industrial']},
                baseline.copy(), {}, 0.0,
            ),
            ScenarioTest(
                'vehicle_breakdown_001', 'Vehicle Breakdown Scenario',
                {'vehicles_down': 1, 'breakdown_duration': 2,
                 'affected_routes': [1, 2]},
                baseline.copy(), {}, 0.0,
            ),
            ScenarioTest(
                'weather_disruption_001', 'Weather Disruption Scenario',
                {'weather_condition': 'heavy_rain', 'visibility_reduction': 0.5,
                 'speed_reduction': 0.3, 'duration_hours': 3},
                baseline.copy(), {}, 0.0,
            ),
            ScenarioTest(
                'route_optimization_001', 'Route Optimization Scenario',
                {'optimization_method': '2-opt', 'improvement_target': 0.15,
                 'computation_time_limit': 300},
                baseline.copy(), {}, 0.0,
            ),
        ]

    def run_scenario_test(
        self, scenario: ScenarioTest, current_routes: List[List[int]]
    ) -> Dict[str, Any]:
        """Evaluate one scenario and record its simulated duration."""
        start_time = time.perf_counter()
        results = self._simulate_scenario(scenario, current_routes)
        scenario.test_duration = time.perf_counter() - start_time
        scenario.test_results = results
        baseline_distance = scenario.baseline_metrics['total_distance']
        test_distance = results.get('total_distance', baseline_distance)
        scenario.improvement_percentage = (
            (baseline_distance - test_distance) / baseline_distance
            if baseline_distance else 0.0
        )
        self.test_results.append(results)
        return results

    def _simulate_scenario(
        self, scenario: ScenarioTest, current_routes: List[List[int]]
    ) -> Dict[str, Any]:
        """Dispatch to the arithmetic rule associated with the scenario."""
        dispatch = {
            'Demand Surge Scenario': self._simulate_demand_surge,
            'Vehicle Breakdown Scenario': self._simulate_vehicle_breakdown,
            'Weather Disruption Scenario': self._simulate_weather_disruption,
            'Route Optimization Scenario': self._simulate_route_optimization,
        }
        simulator = dispatch.get(scenario.scenario_name)
        return simulator(scenario, current_routes) if simulator else scenario.baseline_metrics.copy()

    def _simulate_demand_surge(
        self, scenario: ScenarioTest, current_routes: List[List[int]]
    ) -> Dict[str, Any]:
        multiplier = scenario.parameters['demand_multiplier']
        baseline = scenario.baseline_metrics
        return {
            'total_distance': baseline['total_distance'] * (1 + 0.2 * multiplier),
            'delivery_time': baseline['delivery_time'] * (1 + 0.3 * multiplier),
            'fuel_consumption': baseline['fuel_consumption'] * (1 + 0.15 * multiplier),
            'vehicle_utilization': min(1.0, baseline['vehicle_utilization'] * multiplier),
            'additional_vehicles_needed': max(0, int(multiplier - 1)),
        }

    def _simulate_vehicle_breakdown(
        self, scenario: ScenarioTest, current_routes: List[List[int]]
    ) -> Dict[str, Any]:
        vehicles_down = scenario.parameters['vehicles_down']
        baseline = scenario.baseline_metrics
        return {
            'total_distance': baseline['total_distance'] * (1 + 0.25 * vehicles_down),
            'delivery_time': baseline['delivery_time'] * (1 + 0.4 * vehicles_down),
            'fuel_consumption': baseline['fuel_consumption'] * (1 + 0.1 * vehicles_down),
            'vehicle_utilization': min(
                1.0, baseline['vehicle_utilization'] * (1 + 0.3 * vehicles_down)
            ),
            'service_disruption': 0.1 * vehicles_down,
        }

    def _simulate_weather_disruption(
        self, scenario: ScenarioTest, current_routes: List[List[int]]
    ) -> Dict[str, Any]:
        speed_reduction = scenario.parameters['speed_reduction']
        baseline = scenario.baseline_metrics
        return {
            'total_distance': baseline['total_distance'] * (1 + 0.1 * speed_reduction),
            'delivery_time': baseline['delivery_time'] * (1 + speed_reduction),
            'fuel_consumption': baseline['fuel_consumption'] * (1 + 0.2 * speed_reduction),
            'vehicle_utilization': baseline['vehicle_utilization'] * (1 - 0.1 * speed_reduction),
            'safety_incidents': 0.05 * speed_reduction,
        }

    def _simulate_route_optimization(
        self, scenario: ScenarioTest, current_routes: List[List[int]]
    ) -> Dict[str, Any]:
        """Apply the notebook's 2-opt procedure and report proxy-distance change."""
        optimizer = PredictiveAnalyticsSystem()
        original_distance = sum(
            optimizer._calculate_route_distance(route) for route in current_routes
        )
        optimized_routes = [
            optimizer._two_opt_optimization(route) for route in current_routes
        ]
        optimized_distance = sum(
            optimizer._calculate_route_distance(route) for route in optimized_routes
        )
        baseline = scenario.baseline_metrics
        distance_ratio = (
            optimized_distance / original_distance if original_distance else 1.0
        )
        target = scenario.parameters['improvement_target']
        return {
            'total_distance': baseline['total_distance'] * distance_ratio,
            'delivery_time': baseline['delivery_time'] * distance_ratio,
            'fuel_consumption': baseline['fuel_consumption'] * distance_ratio,
            'vehicle_utilization': baseline['vehicle_utilization'],
            'computation_time_limit_seconds': scenario.parameters['computation_time_limit'],
        }

    def get_scenario_summary(self) -> Dict[str, Any]:
        """Summarize completed tests without treating elapsed time as speed evidence."""
        completed = [scenario for scenario in self.scenarios if scenario.test_results]
        summary: Dict[str, Any] = {
            'total_scenarios': len(self.scenarios),
            'completed_tests': len(completed),
            'average_improvement': 0.0,
            'best_scenario': None,
            'test_speed': 0.0,
        }
        if completed:
            summary['average_improvement'] = float(np.mean(
                [scenario.improvement_percentage for scenario in completed]
            ))
            best = max(completed, key=lambda scenario: scenario.improvement_percentage)
            summary['best_scenario'] = {
                'name': best.scenario_name,
                'improvement': best.improvement_percentage,
            }
            total_duration = sum(scenario.test_duration for scenario in completed)
            if total_duration > 0:
                summary['test_speed'] = len(completed) / total_duration
        return summary

print('Scenario testing system defined.')

## Reading the model and its outputs

The sensor manager creates observations; the integration layer selects the latest available observation from each sensor and organizes it by vehicle, location, or customer. A data stream is a configuration record in this example, not a live network connection. Its rate, latency, and reliability fields are configured inputs.

The predictive section contains three distinct demonstrations. Demand is generated from a sinusoidal daily pattern plus seeded random variation; the `historical_data` argument is retained as an explicit interface but is not fitted or analyzed. Fuel use is an illustrative per-customer estimate adjusted by weather and congestion. Route ordering uses 2-opt: it repeatedly reverses a section of a route when that change lowers the defined distance.

For a route \(r=(r_1,\ldots,r_m)\), the distance proxy is

$$
D(r)=|r_1-0|+\sum_{k=1}^{m-1}|r_{k+1}-r_k|+|0-r_m|.
$$

Customer identifier \(r_k\) is treated as a one-dimensional coordinate, and \(0\) is the depot. The 2-opt search considers reversals of consecutive route sections and keeps an improving ordering. Its role here is a local route-ordering demonstration, not a CVRP solver: there are no capacity constraints, road distances, time windows, or decisions that move customers between routes. The function `_calculate_route_distance` implements \(D(r)\), and `_two_opt_optimization` implements the search. Because this proxy is a line, reversing a route section may leave its distance unchanged; a zero improvement is a valid outcome.

The scenario layer computes changes from stated baseline metrics using explicit arithmetic rules. Its reported improvement is the relative reduction in the scenario's `total_distance` compared with that scenario's baseline:

$$
\text{improvement}=\frac{D_{\text{baseline}}-D_{\text{scenario}}}{D_{\text{baseline}}},
$$

when the baseline distance is nonzero. A negative value means the scenario's distance is higher than its baseline; it does not mean the simulated operation has been experimentally measured. Scenario duration is the computer time needed to execute these simple calculations, not a benchmark of full-scale scenario simulation.

In [ ]:
class DigitalTwinCVRPSystem:
    """Connect sensor, integration, analytics, and scenario components."""

    def __init__(self) -> None:
        self.sensor_manager = IoTSensorManager()
        self.data_integration = RealTimeDataIntegration(self.sensor_manager)
        self.predictive_analytics = PredictiveAnalyticsSystem()
        self.scenario_testing = ScenarioTestingSystem()
        self.digital_twin = DigitalTwin(
            twin_id='cvrp_digital_twin_001',
            physical_system_id='cvrp_physical_001',
            synchronization_rate=1.0,
            fidelity_level=0.95,
            uptime=99.7,
        )
        self.digital_twin.sensors = self.sensor_manager.sensors
        self.digital_twin.data_streams = self.data_integration.data_streams
        self.digital_twin.predictive_models = self.predictive_analytics.models
        self.digital_twin.scenario_tests = self.scenario_testing.scenarios
        self.system_history: List[Dict[str, Any]] = []

    def initialize_digital_twin(self) -> Dict[str, Any]:
        """Generate the initial synthetic observations and assemble a snapshot."""
        print('=' * 70)
        print('DIGITAL-TWIN CVRP DEMONSTRATION: INITIALIZATION')
        print('=' * 70)
        print('\nGenerating synthetic sensor measurements...')
        self.sensor_manager.generate_sensor_data(duration_minutes=10)
        sensor_stats = self.sensor_manager.get_sensor_statistics()
        print('\nSensor network configuration:')
        print(f"  Total sensors: {sensor_stats['total_sensors']}")
        print(f"  Active sensors: {sensor_stats['active_sensors']}")
        print(f"  Generated measurements: {sensor_stats['total_data_points']}")
        print(f"  Mean configured sensor accuracy: {sensor_stats['average_accuracy']:.1%}")
        print(f"  Simulated throughput: {sensor_stats['data_points_per_minute']:.1f} points/minute")
        print('\nSensor types:')
        for sensor_type, count in sensor_stats['sensor_types'].items():
            print(f'  {sensor_type.title()}: {count}')

        real_time_data = self.data_integration.process_real_time_data()
        sync_metrics = self.data_integration.calculate_synchronization_metrics()
        print('\nLatest-observation snapshot:')
        print(f"  Vehicle positions: {len(real_time_data['vehicle_positions'])}")
        print(f"  Fuel readings: {len(real_time_data['fuel_levels'])}")
        print(f"  Load readings: {len(real_time_data['load_weights'])}")
        print(f"  Environmental locations: {len(real_time_data['environmental_conditions'])}")
        print(f"  Customer locations: {len(real_time_data['customer_locations'])}")
        print('\nConfigured synchronization indicators:')
        print(f"  Mean configured sensor accuracy: {sync_metrics['sync_accuracy']:.1%}")
        print(f"  Active-sensor fraction: {sync_metrics['system_reliability']:.1%}")
        print(f"  Mean configured stream latency: {sync_metrics['avg_latency']:.1f} ms")
        print(f"  Configured uptime example: {sync_metrics['uptime']:.1f}%")

        model_performance = self.predictive_analytics.get_model_performance()
        print('\nConfigured predictive-model records (not validation results):')
        for model_id, performance in model_performance.items():
            print(f"  {model_id}: {performance['model_type']}, configured accuracy {performance['accuracy']:.1%}")
        return {
            'sensor_statistics': sensor_stats,
            'real_time_data': real_time_data,
            'synchronization_metrics': sync_metrics,
            'model_performance': model_performance,
        }

    def run_predictive_analysis(
        self, current_routes: List[List[int]]
    ) -> Dict[str, Any]:
        """Run the illustrative demand, route-ordering, and fuel calculations."""
        print('\n' + '=' * 70)
        print('PREDICTIVE ANALYSIS DEMONSTRATION')
        print('=' * 70)
        demand_forecast = self.predictive_analytics.predict_demand(
            historical_data=[], forecast_horizon=24
        )
        predictions = demand_forecast['predictions']
        print('\nSynthetic demand pattern:')
        print(f"  Forecast horizon: {demand_forecast['forecast_horizon']} hours")
        print(f"  Configured model accuracy: {demand_forecast['model_accuracy']:.1%}")
        print(f'  Mean generated demand: {np.mean(predictions):.1f}')
        print(f'  Maximum generated demand: {max(predictions):.1f}')
        print(f'  Minimum generated demand: {min(predictions):.1f}')

        route_optimization = self.predictive_analytics.predict_optimal_routes(
            current_routes, demand_forecast
        )
        print('\n2-opt route-ordering demonstration:')
        print(f"  Routes supplied: {len(current_routes)}")
        print(f"  Mean proxy-distance improvement: {route_optimization['average_improvement']:.1%}")
        print('  Interpretation: this proxy is not road distance or a capacity-constrained CVRP objective.')

        sample_weather = {'temperature': 15.0, 'wind_speed': 10.0, 'congestion': 0.6}
        fuel_prediction = self.predictive_analytics.predict_fuel_consumption(
            current_routes, sample_weather
        )
        average_fuel = (
            float(np.mean(fuel_prediction['fuel_predictions']))
            if fuel_prediction['fuel_predictions'] else 0.0
        )
        print('\nIllustrative fuel estimate:')
        print(f"  Configured model accuracy: {fuel_prediction['model_accuracy']:.1%}")
        print(f"  Total estimated consumption: {fuel_prediction['total_consumption']:.1f} units")
        print(f'  Mean estimate per route: {average_fuel:.1f} units')
        return {
            'demand_forecast': demand_forecast,
            'route_optimization': route_optimization,
            'fuel_prediction': fuel_prediction,
        }

    def run_scenario_testing(
        self, current_routes: List[List[int]]
    ) -> Dict[str, Any]:
        """Run all configured arithmetic what-if scenarios."""
        print('\n' + '=' * 70)
        print('ILLUSTRATIVE SCENARIO TESTING')
        print('=' * 70)
        test_results = []
        for scenario in self.scenario_testing.scenarios:
            result = self.scenario_testing.run_scenario_test(scenario, current_routes)
            print(f"\n{scenario.scenario_name}:")
            print(f'  Calculation time: {scenario.test_duration:.6f} seconds')
            print(f'  Baseline-relative distance change: {scenario.improvement_percentage:.1%}')
            for metric, value in result.items():
                print(f'  {metric}: {value:.2f}')
            test_results.append({
                'scenario_name': scenario.scenario_name,
                'improvement': scenario.improvement_percentage,
                'duration': scenario.test_duration,
                'results': result,
            })
        scenario_summary = self.scenario_testing.get_scenario_summary()
        print('\nScenario summary:')
        print(f"  Configured scenarios: {scenario_summary['total_scenarios']}")
        print(f"  Completed calculations: {scenario_summary['completed_tests']}")
        print(f"  Mean baseline-relative distance change: {scenario_summary['average_improvement']:.1%}")
        print('  Note: calculation speed is hardware- and session-dependent.')
        return {'test_results': test_results, 'scenario_summary': scenario_summary}

    def generate_digital_twin_report(
        self,
        init_results: Dict[str, Any],
        predictive_results: Dict[str, Any],
        scenario_results: Dict[str, Any],
    ) -> str:
        """Assemble a report whose claims are limited to configured and simulated values."""
        stats = init_results['sensor_statistics']
        sync = init_results['synchronization_metrics']
        demand = predictive_results['demand_forecast']
        routing = predictive_results['route_optimization']
        fuel = predictive_results['fuel_prediction']
        scenarios = scenario_results['scenario_summary']
        return f'''DIGITAL-TWIN CVRP DEMONSTRATION REPORT
=====================================

SYSTEM CONFIGURATION
- Digital twin ID: {self.digital_twin.twin_id}
- Synchronization rate setting: {self.digital_twin.synchronization_rate:.1f} Hz
- Fidelity setting: {self.digital_twin.fidelity_level:.1%} (configured example)
- Uptime setting: {self.digital_twin.uptime:.1f}% (configured example)

SYNTHETIC SENSOR DATA
- Sensors: {stats['total_sensors']}
- Generated measurements: {stats['total_data_points']:,}
- Mean configured sensor accuracy: {stats['average_accuracy']:.1%}
- Simulated throughput: {stats['data_points_per_minute']:.1f} points/minute

INTEGRATION INDICATORS
- Active-sensor fraction: {sync['system_reliability']:.1%}
- Mean configured sensor accuracy: {sync['sync_accuracy']:.1%}
- Mean configured stream latency: {sync['avg_latency']:.1f} ms

ILLUSTRATIVE ANALYTICS
- Demand forecast horizon: {demand['forecast_horizon']} hours
- Demand model accuracy setting: {demand['model_accuracy']:.1%} (not validated here)
- Route-ordering proxy improvement: {routing['average_improvement']:.1%}
- Fuel estimate: {fuel['total_consumption']:.1f} units (illustrative model)

SCENARIO CALCULATIONS
- Configured scenarios: {scenarios['total_scenarios']}
- Completed calculations: {scenarios['completed_tests']}
- Mean baseline-relative distance change: {scenarios['average_improvement']:.1%}

SCOPE
This notebook demonstrates a simulated data and reporting pipeline. It does not connect to IoT devices,
validate predictive accuracy, benchmark full-scale scenario simulation, or solve a capacitated VRP.
Its route-distance proxy is one-dimensional and is not a road-network distance.
'''

print('Main digital-twin demonstration system defined.')

In [ ]:
digital_twin_system = DigitalTwinCVRPSystem()
init_results = digital_twin_system.initialize_digital_twin()

# Four example vehicle routes cover customers 1 through 10 exactly once.
# Empty-route and vehicle-capacity feasibility are not optimization decisions here.
current_routes = [
    [1, 3, 5, 7],
    [2, 4, 6, 8],
    [9, 10],
    [],
]
assigned_customers = [customer for route in current_routes for customer in route]
assert sorted(assigned_customers) == list(range(1, 11))
assert len(assigned_customers) == len(set(assigned_customers))

predictive_results = digital_twin_system.run_predictive_analysis(current_routes)
scenario_results = digital_twin_system.run_scenario_testing(current_routes)
digital_twin_report = digital_twin_system.generate_digital_twin_report(
    init_results, predictive_results, scenario_results
)
print('\n' + digital_twin_report)

In [ ]:
def visualize_digital_twin(
    digital_twin_system: DigitalTwinCVRPSystem,
    init_results: Dict[str, Any],
    predictive_results: Dict[str, Any],
    scenario_results: Dict[str, Any],
) -> None:
    """Plot sensor composition, configured model values, scenarios, and scope notes."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    ax1, ax2, ax3, ax4 = axes.flat

    sensor_types = init_results['sensor_statistics']['sensor_types']
    ax1.pie(
        list(sensor_types.values()),
        labels=list(sensor_types.keys()),
        autopct='%1.1f%%',
        startangle=90,
    )
    ax1.set_title('Configured Sensor Network Composition')
    ax1.axis('equal')

    model_records = digital_twin_system.predictive_analytics.get_model_performance()
    model_labels = [
        'Demand forecast', 'Route ordering', 'Fuel estimate', 'Traffic prediction'
    ]
    model_ids = [
        'demand_forecast_v1', 'route_optimizer_v1',
        'fuel_consumption_v1', 'traffic_predictor_v1',
    ]
    accuracies = [model_records[model_id]['accuracy'] * 100 for model_id in model_ids]
    bars = ax2.bar(model_labels, accuracies, color='steelblue', edgecolor='black')
    ax2.set_title('Configured Accuracy Values (Not Validated)')
    ax2.set_ylabel('Configured value (%)')
    ax2.set_ylim(0, 100)
    ax2.tick_params(axis='x', rotation=20)
    ax2.grid(axis='y', alpha=0.3)
    for bar, accuracy in zip(bars, accuracies):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 f'{accuracy:.1f}%', ha='center', va='bottom', fontsize=9)

    results = scenario_results['test_results']
    names = [item['scenario_name'].replace(' Scenario', '') for item in results]
    changes = [item['improvement'] * 100 for item in results]
    colors = ['seagreen' if value >= 0 else 'indianred' for value in changes]
    bars = ax3.bar(names, changes, color=colors, edgecolor='black')
    ax3.axhline(0, color='black', linewidth=0.8)
    ax3.set_title('Scenario Baseline-Relative Distance Change')
    ax3.set_ylabel('Distance reduction (%)')
    ax3.set_xlabel('Illustrative scenario')
    ax3.tick_params(axis='x', rotation=25)
    ax3.grid(axis='y', alpha=0.3)
    for bar, value in zip(bars, changes):
        offset = 1 if value >= 0 else -1
        ax3.text(bar.get_x() + bar.get_width() / 2, value + offset,
                 f'{value:.1f}%', ha='center',
                 va='bottom' if value >= 0 else 'top', fontsize=9)

    ax4.axis('off')
    stats = init_results['sensor_statistics']
    sync = init_results['synchronization_metrics']
    route = predictive_results['route_optimization']
    summary = scenario_results['scenario_summary']
    dashboard = (
        'SIMULATION SUMMARY\n'
        '------------------\n'
        f"Generated sensor measurements: {stats['total_data_points']:,}\n"
        f"Simulated throughput: {stats['data_points_per_minute']:.1f} points/min\n"
        f"Active-sensor fraction: {sync['system_reliability']:.1%}\n"
        f"Configured mean latency: {sync['avg_latency']:.1f} ms\n\n"
        f"2-opt proxy improvement: {route['average_improvement']:.1%}\n"
        f"Completed scenario calculations: {summary['completed_tests']}\n"
        f"Mean scenario distance reduction: {summary['average_improvement']:.1%}\n\n"
        'Scope: synthetic data and simplified rules.\n'
        'No live devices or CVRP capacity constraints.'
    )
    ax4.text(
        0.03, 0.97, dashboard, fontsize=10, va='top', family='monospace',
        bbox={'boxstyle': 'round', 'facecolor': 'honeydew', 'alpha': 0.8},
    )
    ax4.set_title('Interpretation and Scope')

    fig.suptitle('Digital-Twin Simulation Dashboard', fontsize=15)
    fig.tight_layout()
    plt.show()


visualize_digital_twin(
    digital_twin_system, init_results, predictive_results, scenario_results
)

## Why this tier matters

A digital-twin architecture connects observations, models, scenario analysis, and decision support in a shared representation of an operation. Its value depends on the quality and timeliness of the underlying data, the validity of its models, and the decisions made from their outputs.

### Capabilities represented in this demonstration

- **Data integration:** sensor observations are organized into a snapshot by operational entity.
- **Predictive analysis:** demand and fuel estimates illustrate how a model's output can be presented alongside its assumptions.
- **Scenario analysis:** explicit what-if rules show how changes in assumptions affect reported metrics.
- **Monitoring and reporting:** summaries and plots bring configuration values and simulated measurements together.

These are architectural concepts, not evidence that a live digital twin has been deployed. Real-time synchronization requires live interfaces, timestamp and data-quality handling, monitoring of communication failures, and operational validation. Predictive accuracy requires comparison with held-out observations and suitable evaluation metrics. Scenario conclusions require validated relationships between scenario inputs and operational outcomes.

### Benefits and limitations

**Potential benefits when supported by validated systems**

- Current operational observations can inform timely decisions.
- Forecasts can help planners anticipate changing demand or operating conditions.
- Scenario analysis can make assumptions and potential consequences explicit.
- Shared representations can improve visibility across interconnected operations.

**Important limitations and risks**

- Integration, sensor maintenance, data governance, security, and model monitoring require ongoing resources.
- Missing, delayed, or inaccurate observations can make a virtual representation misleading.
- A model's configured accuracy value is not a substitute for empirical validation.
- Simplified scenario rules do not establish what would happen in a real logistics network.
- A digital-twin architecture does not replace the mathematical formulation and solution of the CVRP.

### When the concepts may be useful

Digital-twin methods may be useful in large, dynamic operations where timely data and validated models can support decisions. Their suitability depends on the value of improved information relative to implementation, maintenance, security, and governance costs. A natural extension of this demonstration would be to connect validated operational data and richer scenario assumptions while keeping those components clearly distinguished from the routing optimization model.